# Integrating Predictive Models with Operational Policies for Constrained Lead Qualification
This notebook reflects the coding for the masters dissertation.

In [9]:

!pip install xgboost -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Setup complete.")



Setup complete.


In [17]:
# Loading data from google drive:
from google.colab import drive
drive.mount('/content/drive')
df_raw = pd.read_csv("/content/drive/MyDrive/Masters_Constrained_Lead_Qualification/Lead Scoring.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
# Upload "Lead Scoring.csv" to Colab, or load from Google Drive.


print(f"Shape: {df_raw.shape}")
print(f"Columns:\n{df_raw.columns.tolist()}")
print(f"\nConversion rate: {df_raw['Converted'].mean():.4f}")
df_raw.head()

Shape: (9240, 37)
Columns:
['Prospect ID', 'Lead Number', 'Lead Origin', 'Lead Source', 'Do Not Email', 'Do Not Call', 'Converted', 'TotalVisits', 'Total Time Spent on Website', 'Page Views Per Visit', 'Last Activity', 'Country', 'Specialization', 'How did you hear about X Education', 'What is your current occupation', 'What matters most to you in choosing a course', 'Search', 'Magazine', 'Newspaper Article', 'X Education Forums', 'Newspaper', 'Digital Advertisement', 'Through Recommendations', 'Receive More Updates About Our Courses', 'Tags', 'Lead Quality', 'Update me on Supply Chain Content', 'Get updates on DM Content', 'Lead Profile', 'City', 'Asymmetrique Activity Index', 'Asymmetrique Profile Index', 'Asymmetrique Activity Score', 'Asymmetrique Profile Score', 'I agree to pay the amount through cheque', 'A free copy of Mastering The Interview', 'Last Notable Activity']

Conversion rate: 0.3854


,Prospect ID,Lead Number,Lead Origin,Lead Source,Do Not Email,Do Not Call,Converted,TotalVisits,Total Time Spent on Website,Page Views Per Visit,...,Get updates on DM Content,Lead Profile,City,Asymmetrique Activity Index,Asymmetrique Profile Index,Asymmetrique Activity Score,Asymmetrique Profile Score,I agree to pay the amount through cheque,A free copy of Mastering The Interview,Last Notable Activity
0,7927b2df-8bba-4d29-b9a2-b6e0beafe620,660737,API,Olark Chat,No,No,0,0.0,0,0.0,...,No,Select,Select,02.Medium,02.Medium,15.0,15.0,No,No,Modified
1,2a272436-5132-4136-86fa-dcc88c88f482,660728,API,Organic Search,No,No,0,5.0,674,2.5,...,No,Select,Select,02.Medium,02.Medium,15.0,15.0,No,No,Email Opened
2,8cc8c611-a219-4f35-ad23-fdfd2656bd8a,660727,Landing Page Submission,Direct Traffic,No,No,1,2.0,1532,2.0,...,No,Potential Lead,Mumbai,02.Medium,01.High,14.0,20.0,No,Yes,Email Opened
3,0cc2df48-7cf4-4e39-9de9-19797f9b38cc,660719,Landing Page Submission,Direct Traffic,No,No,0,1.0,305,1.0,...,No,Select,Mumbai,02.Medium,01.High,13.0,17.0,No,No,Modified
4,3256f628-e534-4826-9d63-4a8b88782852,660681,Landing Page Submission,Google,No,No,1,2.0,1428,1.0,...,No,Select,Mumbai,02.Medium,01.High,15.0,18.0,No,No,Modified


In [19]:
# =============================================================================
# CELL 02 — PRESERVE RAW POLICY COLUMNS
#
# This must happen BEFORE any cleaning, grouping, or encoding.
# Policy rules (H1, H2, P1, P2, S1, S2, S3) are applied using these raw
# values. If you group rare categories first, values like "Referral" or
# "Email Link Clicked" may be collapsed into "Other" and break S3/S4.
# =============================================================================

POLICY_COLUMNS = [
    "Do Not Email",
    "Do Not Call",
    "TotalVisits",
    "Total Time Spent on Website",
    "Last Activity",
    "Lead Origin",
    "Lead Source",
    "What is your current occupation",
]

# Confirm all policy columns exist in the raw data
missing_policy = [c for c in POLICY_COLUMNS if c not in df_raw.columns]
if missing_policy:
    raise ValueError(f"Missing policy columns in raw data: {missing_policy}")

policy_df_full = df_raw[POLICY_COLUMNS].copy()

print("Raw policy columns preserved.")
print(f"\nDo Not Email value counts:\n{policy_df_full['Do Not Email'].value_counts(dropna=False)}")
print(f"\nDo Not Call value counts:\n{policy_df_full['Do Not Call'].value_counts(dropna=False)}")

Raw policy columns preserved.

Do Not Email value counts:
Do Not Email
No     8506
Yes     734
Name: count, dtype: int64

Do Not Call value counts:
Do Not Call
No     9238
Yes       2
Name: count, dtype: int64


In [20]:

# =============================================================================
# CELL 03 — LEAKAGE REMOVAL
#
# These columns are excluded a priori, before any data inspection of their
# predictive value. They contain post-hoc human judgments or derived scores
# that are correlated with the conversion outcome and would not be available
# at the point of real-time qualification.
# =============================================================================

LEAKAGE_COLUMNS = [
    "Tags",
    "Lead Quality",
    "Asymmetrique Activity Index",
    "Asymmetrique Profile Index",
    "Asymmetrique Activity Score",
    "Asymmetrique Profile Score",
]

ID_COLUMNS = ["Prospect ID", "Lead Number"]

cols_to_drop = [c for c in LEAKAGE_COLUMNS + ID_COLUMNS if c in df_raw.columns]
df = df_raw.drop(columns=cols_to_drop).copy()

print(f"Dropped {len(cols_to_drop)} columns: {cols_to_drop}")
print(f"Shape after leakage removal: {df.shape}")



Dropped 8 columns: ['Tags', 'Lead Quality', 'Asymmetrique Activity Index', 'Asymmetrique Profile Index', 'Asymmetrique Activity Score', 'Asymmetrique Profile Score', 'Prospect ID', 'Lead Number']
Shape after leakage removal: (9240, 29)


In [21]:

# =============================================================================
# CELL 04 — CLEAN DATA
#
# Steps:
#   a. Replace "Select" placeholder with NaN (form fields left blank by user)
#   b. Remove duplicates
#   c. Handle high-missing columns
#   d. Construct derived feature: Engagement Intensity
#   e. Note: Do Not Email / Do Not Call are NOT used as model features.
#      They are kept in the dataframe here only so they survive the split
#      alongside X. Policy rules consume them from policy_df, not from X.
# =============================================================================

# a. Replace "Select" with NaN
df = df.replace("Select", np.nan)

# b. Remove duplicates
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate rows.")

# c. Drop columns with >40% missing values
missing_pct = df.isnull().mean() * 100
high_missing = missing_pct[missing_pct > 40].index.tolist()
print(f"\nColumns with >40% missing (will be dropped): {high_missing}")
df = df.drop(columns=high_missing)

Removed 1687 duplicate rows.

Columns with >40% missing (will be dropped): ['How did you hear about X Education', 'Lead Profile']


In [22]:

# d. Derived feature: average depth of engagement per visit
#    Captures intensity of interest rather than raw visit volume.
#    A lead with 60 minutes across 2 visits differs from one with 60 minutes
#    across 20 visits. Computed before splitting; used as a model feature only.
df["Engagement Intensity"] = (
    df["Total Time Spent on Website"] / df["TotalVisits"].replace(0, np.nan)
)
df["Engagement Intensity"] = df["Engagement Intensity"].replace(
    [np.inf, -np.inf], np.nan
)

print(f"\nShape after cleaning: {df.shape}")
print(f"\nRemaining missing values (top 10):")
print(df.isnull().sum().sort_values(ascending=False).head(10))



Shape after cleaning: (7553, 28)

Remaining missing values (top 10):
City                                             2040
What matters most to you in choosing a course    1921
What is your current occupation                  1902
Specialization                                   1844
Country                                           779
Engagement Intensity                              644
Page Views Per Visit                              136
TotalVisits                                       136
Last Activity                                     102
Lead Source                                        23
dtype: int64
